In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib seaborn

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Φόρτωση συνόλου δεδομένων και speaker independent διαχωρισμός

In [ ]:
#Φόρτωση του dataset, από το HuggingFace
dataset = load_dataset("AbstractTTS/CREMA-D", split="train")

#Μετατροπή συχνότητας σε 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

print(dataset.features)
print(dataset[0]["file"], dataset[0]["major_emotion"])

In [ ]:
labels = sorted(set(dataset["major_emotion"]))
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}

def add_speaker(batch):
    batch["actor_id"] = batch["file"].split("_")[0]
    return batch

dataset = dataset.map(add_speaker, load_from_cache_file=False)
print(len(set(dataset["actor_id"])), "μοναδικοί ομιλητές")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

actor_ids = dataset["actor_id"]
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=actor_ids))

from datasets import DatasetDict
dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx)
})

# Έλεγχος ότι δεν υπάρχει επικάλυψη ομιλητών
train_speakers = set(dataset["train"]["actor_id"])
test_speakers = set(dataset["test"]["actor_id"])
print("Επικάλυψη:", train_speakers & test_speakers)

### Φόρτωση του αρχικού προεκπαιδευμένου μοντέλου

In [ ]:
model_repo = "MIT/ast-finetuned-audioset-10-10-0.4593"

#Φόρτωση του feature extractor του βασικού μοντέλου
feature_extractor = ASTFeatureExtractor.from_pretrained(model_repo)

#Φόρτωση του προεκπαιδευμένου μοντέλου με νέα κεφαλή ταξινόμησης για τις 6 κλάσεις
model = ASTForAudioClassification.from_pretrained(
    model_repo,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
).to("cuda")

In [ ]:
MAX_LENGTH = 6 * 16000

keep_cols = ["audio", "major_emotion"]
dataset = dataset.remove_columns(
    [c for c in dataset["train"].column_names if c not in keep_cols]
)
print(dataset)

### Ορισμός μετρικών αξιολόγησης

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,

    average_precision_score
)

from sklearn.preprocessing import label_binarize
from scipy.special import softmax

In [ ]:
#Λίστα για την αποθήκευση των confusion matrices
all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)
    labels_bin = label_binarize(labels_true, classes=list(range(num_labels)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")

    # PR-AUC (macro-averaged average precision)
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    #Σταθερό σχήμα πίνακα ακόμα κι αν κάποια κλάση δεν προβλεφθεί καθόλου
    cm = confusion_matrix(labels_true, predictions, labels=list(range(num_labels)))
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy_score(labels_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels_true, predictions),
        "precision": precision_score(labels_true, predictions, average="weighted", zero_division=0),
        "recall": recall_score(labels_true, predictions, average="weighted", zero_division=0),
        "f1": f1_score(labels_true, predictions, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(labels_true, predictions),
        "k_score": cohen_kappa_score(labels_true, predictions),
        "log_loss": log_loss(labels_true, probs, labels=list(range(num_labels))),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

###Data Collator

In [ ]:
#Data Collator: περικοπή στα 6s, μετατροπή σε log-mel filterbank και μετατροπή ετικετών, ανά batch
class DataCollatorASTOnTheFly:
    def __init__(self, feature_extractor, label2id, max_length):
        self.feature_extractor = feature_extractor
        self.label2id = label2id
        self.max_length = max_length

    def __call__(self, features):
        #Περικοπή στα 6 δευτερόλεπτα πριν τον feature extractor (αυτός συμπληρώνει πάντα στα 1024 frames)
        audio_arrays = [f["audio"]["array"][:self.max_length] for f in features]

        #Μετατροπή σε κανονικοποιημένο log-mel filterbank, σχήμα (batch, 1024, 128)
        inputs = self.feature_extractor(
            audio_arrays,
            sampling_rate=16000,
            return_tensors="pt",
        )

        #Μετατροπή της ετικέτας από string σε integer id, για συμβατότητα με τον trainer
        labels = torch.tensor(
            [self.label2id[f["major_emotion"]] for f in features], dtype=torch.long
        )
        return {"input_values": inputs["input_values"], "labels": labels}

data_collator = DataCollatorASTOnTheFly(feature_extractor, label2id, MAX_LENGTH)

#Έλεγχος σε ένα μικρό batch: πρέπει (4, 1024, 128) και 4 ετικέτες
test_batch = data_collator([dataset["train"][i] for i in range(4)])
print(test_batch["input_values"].shape, test_batch["labels"])

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
# Δυναμικός υπολογισμός warmup_steps (10% των συνολικών training steps)
per_device_train_batch_size = 8
gradient_accumulation_steps = 4
num_train_epochs = 8

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = len(dataset["train"]) // effective_batch_size   # αντί για encoded_dataset
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))

print(f"Steps/epoch: {steps_per_epoch}, Total steps: {total_steps}, Warmup steps: {warmup_steps}")

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/SER_Thesis/ast_crema_d_checkpoints",
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=False,   # αποφυγή αστάθειας μικτής ακρίβειας, όπως και στα υπόλοιπα μοντέλα
    learning_rate=2e-5,      # όπως στο AST+RAVDESS (μικρότερο, καθώς το checkpoint είναι ήδη fine-tuned στο AudioSet)
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
    remove_unused_columns=False,   # απαραίτητο: αλλιώς ο Trainer αφαιρεί τη στήλη audio πριν φτάσει στον collator
    dataloader_num_workers=2,      # παράλληλη αποκωδικοποίηση ήχου, ώστε η GPU να μην περιμένει
)

In [ ]:
#Καθαρισμός των confusion matrix, από τυχόν προηγούμενες εκτελέσεις
all_confusion_matrices.clear()

### Δημιουργία και Εκπαίδευση του Trainer

In [ ]:
#Κλήση του trainer και εισαγωγή των παραμέτρων εκπαίδευσης
early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

train_dataset=dataset["train"],
eval_dataset=dataset["test"],

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

print("Εκκίνηση εκπαίδε υσης...")
trainer.train()
print("Εκπαίδευση ολοκληρώθηκε!")

In [ ]:
#Επαναφόρτωση του καλύτερου checkpoint μέσω from_pretrained, καθώς το load_best_model_at_end
#δεν αντιστοιχίζει σωστά τα ονόματα των βαρών του AST στη νέα έκδοση του transformers
best_ckpt = trainer.state.best_model_checkpoint
print("Καλύτερο checkpoint:", best_ckpt)

model = ASTForAudioClassification.from_pretrained(best_ckpt).to("cuda")
trainer.model = model

#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

### Οπτικοποίηση αποτελεσμάτων (Confusion Matrices)

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

### Αποθήκευση εκπαιδευμένου μοντέλου στο HuggingFace Hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/AST-finetuned-CREMA_D"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")